In [ ]:
https://github.com/24110207-Hai-Nguyen/8_puzzle_complex

In [ ]:
import tkinter as tk
from tkinter import ttk, messagebox
import random
import collections
import heapq
import time
import threading
import math
from itertools import permutations

# --------------------------- 8-PUZZLE CONFIGURATION ---------------------------
GOAL = (1, 2, 3, 4, 5, 6, 7, 8, 0)

class Node:
    _id_counter = 65
    def __init__(self, state, parent=None, action=None, cost=0, depth=0, h_cost=0):
        self.state = state
        self.parent = parent
        self.action = action
        self.cost = cost
        self.depth = depth
        self.h_cost = h_cost
        self.f_cost = cost + h_cost
        if parent is None:
            self.name = "A"
        else:
            self.name = chr(Node._id_counter)
            Node._id_counter += 1
            if Node._id_counter > 90:
                Node._id_counter = 65
    def __lt__(self, other):
        return self.f_cost < other.f_cost

def heuristic(state):
    return sum(1 for i in range(9) if state[i] != 0 and state[i] != GOAL[i])

def value(state):
    return 9 - heuristic(state)

def get_neighbors(state):
    idx = state.index(0)
    row, col = divmod(idx, 3)
    moves = [(-1, 0, 'Up'), (1, 0, 'Down'), (0, -1, 'Left'), (0, 1, 'Right')]
    neighbors = []
    for dr, dc, move in moves:
        r, c = row + dr, col + dc
        if 0 <= r < 3 and 0 <= c < 3:
            new_idx = r * 3 + c
            new_state = list(state)
            new_state[idx], new_state[new_idx] = new_state[new_idx], new_state[idx]
            neighbors.append((tuple(new_state), move))
    return neighbors

def format_state_matrix(state):
    rows = []
    for i in range(3):
        row = [str(x) if x != 0 else "_" for x in state[i*3:(i+1)*3]]
        rows.append(" " + " ".join(row))
    return "\n".join(rows)

# ---------- Complex mode utilities ----------
def count_inversions(state):
    inv = 0
    for i in range(9):
        for j in range(i+1, 9):
            if state[i] != 0 and state[j] != 0 and state[i] > state[j]:
                inv += 1
    return inv

def is_solvable(state):
    return count_inversions(state) % 2 == 0

def generate_random_solvable_states(num):
    states = []
    while len(states) < num:
        state = list(GOAL)
        for _ in range(30):
            neighbors = get_neighbors(tuple(state))
            state = list(random.choice(neighbors)[0])
        if is_solvable(state) and tuple(state) not in states:
            states.append(tuple(state))
    return states

def parse_template(template_str):
    parts = template_str.strip().split()
    if len(parts) != 9:
        return []
    template = []
    unknown_indices = []
    present_numbers = set()
    for i, p in enumerate(parts):
        if p == '?':
            template.append('?')
            unknown_indices.append(i)
        else:
            try:
                num = int(p)
                if num < 0 or num > 8 or num in present_numbers:
                    return []
                present_numbers.add(num)
                template.append(num)
            except:
                return []
    missing_numbers = [x for x in range(9) if x not in present_numbers]
    if len(missing_numbers) != len(unknown_indices):
        return []
    if len(unknown_indices) > 5:
        messagebox.showwarning("Warning", "Too many '?' (max 5) to avoid performance issues.")
        return []
    possible = []
    for perm in permutations(missing_numbers):
        state_list = template[:]
        for idx, num in zip(unknown_indices, perm):
            state_list[idx] = num
        state = tuple(state_list)
        if is_solvable(state):
            possible.append(state)
    return possible

class ComplexProblem:
    def __init__(self, initial_belief, goal_test, actions_func, results_func):
        self.initial_belief = initial_belief
        self.goal_test = goal_test
        self.actions = actions_func
        self.results = results_func
    def is_goal(self, state):
        return self.goal_test(state)

def and_or_graph_search(problem, max_depth=50, log_func=None):
    def or_search(state, path, depth):
        if log_func:
            log_func(f"OR_SEARCH state={state}, depth={depth}")
        if problem.is_goal(state):
            if log_func: log_func(f"Goal! {state}")
            return []
        if state in path:
            if log_func: log_func(f"Cycle {state}")
            return None
        if depth <= 0:
            return None
        for action in problem.actions(state):
            result_states = problem.results(state, action)
            if log_func:
                log_func(f"  Action {action} -> {result_states}")
            plan = and_search(result_states, path + [state], depth - 1)
            if plan is not None:
                return [action, plan]
        return None

    def and_search(states, path, depth):
        if log_func:
            log_func(f"AND_SEARCH states={states}, depth={depth}")
        if not states:
            return {}
        plans = {}
        for s in states:
            plan_s = or_search(s, path, depth)
            if plan_s is None:
                return None
            plans[s] = plan_s
        return plans

    return and_search(problem.initial_belief, [], max_depth)

def format_plan(plan, indent=0):
    if plan is None:
        return "Failure"
    if isinstance(plan, dict):
        lines = ["AND:"]
        for state, subplan in plan.items():
            lines.append("  " * indent + f"State {state}:")
            lines.append(format_plan(subplan, indent + 1))
        return "\n".join(lines)
    elif isinstance(plan, list):
        if not plan:
            return "  " * indent + "GOAL"
        action = plan[0]
        subplan = plan[1]
        return "  " * indent + f"Action: {action}\n" + format_plan(subplan, indent)
    else:
        return "  " * indent + str(plan)

# --------------------------- MAIN GUI ---------------------------
class EightPuzzleGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("8-Puzzle Search Algorithms")
        self.root.geometry("1300x850")
        self.root.configure(bg="#f5f6f8")
        self.current_state = list(GOAL)
        self.path = []
        self.current_step_idx = 0
        self.is_running = False
        self.complex_submode = tk.StringVar(value="Partial Start, Unknown Goal")
        self.complex_initial_belief = []
        self.complex_goal_states = [GOAL]
        self.conditional_plan = None
        self.setup_ui()
        self.reset_puzzle()

    def setup_ui(self):
        # Top bar
        top_frame = tk.Frame(self.root, bg="#f5f6f8", height=50)
        top_frame.pack(side=tk.TOP, fill=tk.X, padx=20, pady=10)

        tk.Label(top_frame, text="8-PUZZLE SEARCH ALGORITHMS", font=("Arial", 16, "bold"),
                 bg="#f5f6f8", fg="#1e293b").pack(side=tk.LEFT)

        tk.Label(top_frame, text="Group:", bg="#f5f6f8", font=("Arial", 10)).pack(side=tk.LEFT, padx=(20,5))
        self.group_var = tk.IntVar(value=1)
        self.group_menu = ttk.Combobox(top_frame, textvariable=self.group_var, values=list(range(1,7)),
                                       state="readonly", width=3)
        self.group_menu.pack(side=tk.LEFT, padx=5)
        self.group_menu.bind("<<ComboboxSelected>>", self.on_group_change)

        group_names = ["1. Uninformed", "2. Informed", "3. Local", "4. Complex", "5. CSP", "6. Adversarial"]
        self.group_name_var = tk.StringVar(value=group_names[0])
        tk.Label(top_frame, textvariable=self.group_name_var, bg="#f5f6f8", font=("Arial", 10, "bold"),
                 fg="#1e40af").pack(side=tk.LEFT, padx=5)

        # Algorithm frame (groups 1,2,3,5,6)
        self.algo_frame = tk.Frame(top_frame, bg="#f5f6f8")
        self.algo_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)
        tk.Label(self.algo_frame, text="Algorithm:", bg="#f5f6f8").pack(side=tk.LEFT, padx=(10,5))
        self.algo_var = tk.StringVar(value="BFS")
        self.algo_menu = ttk.Combobox(self.algo_frame, textvariable=self.algo_var, values=[], width=25, state="readonly")
        self.algo_menu.pack(side=tk.LEFT, padx=5)
        tk.Label(self.algo_frame, text="Parameter:", bg="#f5f6f8").pack(side=tk.LEFT, padx=(10,5))
        self.param_entry = tk.Spinbox(self.algo_frame, from_=1, to=200, width=5)
        self.param_entry.delete(0, "end"); self.param_entry.insert(0, "35")
        self.param_entry.pack(side=tk.LEFT, padx=5)
        self.btn_random = tk.Button(self.algo_frame, text="Random", bg="#ffffff", bd=1,
                                    relief=tk.SOLID, command=self.shuffle_puzzle, padx=10)
        self.btn_random.pack(side=tk.LEFT, padx=5)
        self.btn_run = tk.Button(self.algo_frame, text="Run", bg="#2563eb", fg="white", bd=0,
                                 relief=tk.SOLID, command=self.start_solve, padx=15)
        self.btn_run.pack(side=tk.LEFT, padx=5)

        # Complex frame (group 4)
        self.complex_frame = tk.Frame(top_frame, bg="#f5f6f8")
        self.complex_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)
        self.complex_frame.pack_forget()
        self.setup_complex_frame()

        self.btn_reset = tk.Button(top_frame, text="Reset", bg="#ffffff", bd=1,
                                   relief=tk.SOLID, command=self.reset_puzzle, padx=10)
        self.btn_reset.pack(side=tk.RIGHT, padx=5)

        # Main layout
        main_frame = tk.Frame(self.root, bg="#f5f6f8")
        main_frame.pack(fill=tk.BOTH, expand=True, padx=20, pady=5)

        # Left column (rearranged)
        left_column = tk.Frame(main_frame, bg="#f5f6f8")
        left_column.pack(side=tk.LEFT, fill=tk.Y, padx=(0,20))

        # --- Top row: Goal box (left) and Puzzle grid (right) ---
        top_row = tk.Frame(left_column, bg="#f5f6f8")
        top_row.pack(fill=tk.X, pady=10)

        # Goal box on the left
        self.goal_box = tk.LabelFrame(top_row, text="Goal State", bg="#ffffff", font=("Arial", 9))
        self.goal_box.pack(side=tk.LEFT, padx=(0,10), pady=5)
        goal_lbl = tk.Label(self.goal_box, text="1  2  3\n4  5  6\n7  8  _", font=("Courier", 12, "bold"),
                            bg="#ffffff", justify=tk.LEFT, padx=10, pady=10)
        goal_lbl.pack()

        # Puzzle grid on the right
        self.grid_frame = tk.Frame(top_row, bg="#f5f6f8")
        self.grid_frame.pack(side=tk.LEFT, padx=10)
        self.cells = []
        for i in range(9):
            btn = tk.Button(self.grid_frame, text="", font=("Arial", 22, "bold"), width=4, height=2,
                            bg="#2563eb", fg="white", bd=0, highlightthickness=0)
            btn.grid(row=i//3, column=i%3, padx=4, pady=4)
            self.cells.append(btn)

        # Navigation, speed, etc. below top row
        nav_frame = tk.Frame(left_column, bg="#f5f6f8")
        nav_frame.pack(fill=tk.X, pady=5)
        self.btn_prev = tk.Button(nav_frame, text="◀ Prev", font=("Arial", 9), command=self.prev_step,
                                  bg="#ffffff", bd=1, relief=tk.SOLID)
        self.btn_prev.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        self.btn_next = tk.Button(nav_frame, text="Next ▶", font=("Arial", 9), command=self.next_step,
                                  bg="#ffffff", bd=1, relief=tk.SOLID)
        self.btn_next.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)
        self.btn_auto = tk.Button(nav_frame, text="Auto Play", font=("Arial", 9), command=self.toggle_auto,
                                  bg="#ffffff", bd=1, relief=tk.SOLID)
        self.btn_auto.pack(side=tk.LEFT, expand=True, fill=tk.X, padx=2)

        speed_frame = tk.Frame(left_column, bg="#f5f6f8")
        speed_frame.pack(fill=tk.X, pady=5)
        tk.Label(speed_frame, text="Speed", bg="#f5f6f8", font=("Arial", 9)).pack(side=tk.LEFT)
        self.speed_scale = tk.Scale(speed_frame, from_=100, to=1000, orient=tk.HORIZONTAL,
                                    bg="#f5f6f8", bd=0, highlightthickness=0)
        self.speed_scale.set(300)
        self.speed_scale.pack(side=tk.RIGHT, fill=tk.X, expand=True, padx=(5,0))

        # Complex panel (hidden initially)
        self.complex_panel = tk.LabelFrame(left_column, text="Complex Environment Settings", bg="#f5f6f8",
                                           font=("Arial", 9))
        self.complex_panel.pack_forget()
        self.setup_complex_panel()

        # Right column
        right_column = tk.Frame(main_frame, bg="#f5f6f8")
        right_column.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)

        info_frame = tk.Frame(right_column, bg="#f5f6f8")
        info_frame.pack(fill=tk.X, pady=(0,15))
        self.stats = {
            "Step": tk.StringVar(value="-"),
            "Algorithm": tk.StringVar(value="-"),
            "Expanded": tk.StringVar(value="-"),
            "Action": tk.StringVar(value="-"),
            "Path Cost": tk.StringVar(value="-"),
            "Depth": tk.StringVar(value="-"),
            "IDS Limit": tk.StringVar(value="-"),
            "Frontier": tk.StringVar(value="-"),
            "Explored": tk.StringVar(value="-"),
            "Status": tk.StringVar(value="Ready.")
        }
        keys = list(self.stats.keys())
        for idx, key in enumerate(keys):
            f = tk.Frame(info_frame, bg="#f5f6f8")
            if key == "Status":
                f.grid(row=9, column=0, columnspan=2, sticky="w", pady=4)
                tk.Label(f, text=f"{key}:", font=("Arial", 10, "bold"), bg="#f5f6f8",
                         width=14, anchor="w").pack(side=tk.LEFT)
                tk.Label(f, textvariable=self.stats[key], font=("Arial", 10, "italic"),
                         bg="#f5f6f8", fg="#475569").pack(side=tk.LEFT)
            else:
                f.grid(row=idx, column=0, sticky="w", pady=2)
                tk.Label(f, text=f"{key}:", font=("Arial", 10), bg="#f5f6f8",
                         width=17, anchor="w").pack(side=tk.LEFT)
                tk.Label(f, textvariable=self.stats[key], font=("Arial", 10, "bold"),
                         bg="#f5f6f8").pack(side=tk.LEFT)

        self.notebook = ttk.Notebook(right_column)
        self.notebook.pack(fill=tk.BOTH, expand=True)
        self.tab_nodes = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)   # renamed from tab_children
        self.tab_frontier = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_explored = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_steplog = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_conditional = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0)
        self.tab_solution = tk.Text(self.notebook, font=("Courier", 10), bg="white", bd=0, cursor="hand2")
        self.tab_solution.tag_configure("highlight", background="#dbeafe")
        self.tab_solution.bind("<Button-1>", self.on_solution_click)

        self.notebook.add(self.tab_nodes, text="Nodes")          # Label changed to "Nodes"
        self.notebook.add(self.tab_frontier, text="Frontier")
        self.notebook.add(self.tab_explored, text="Explored")
        self.notebook.add(self.tab_steplog, text="Step Log")
        self.notebook.add(self.tab_conditional, text="Conditional Plan")
        self.notebook.add(self.tab_solution, text="Solution Path")

        self.update_algo_menu()
        self.configure_tabs_for_group(1)

    def on_group_change(self, event=None):
        g = self.group_var.get()
        group_names = ["1. Uninformed", "2. Informed", "3. Local", "4. Complex", "5. CSP", "6. Adversarial"]
        self.group_name_var.set(group_names[g-1])
        if g == 4:
            self.algo_frame.pack_forget()
            self.complex_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)
            self.complex_panel.pack(fill=tk.X, pady=5, after=self.speed_scale.master)
            self.setup_complex_frame()
            self.setup_complex_panel()
        else:
            self.complex_frame.pack_forget()
            self.complex_panel.pack_forget()
            self.algo_frame.pack(side=tk.LEFT, fill=tk.X, expand=True)
            self.update_algo_menu()
        self.configure_tabs_for_group(g)

    def configure_tabs_for_group(self, g):
        idx_nodes = self.notebook.index(self.tab_nodes)
        idx_frontier = self.notebook.index(self.tab_frontier)
        idx_explored = self.notebook.index(self.tab_explored)

        if g == 3:
            self.notebook.tab(idx_frontier, state="hidden")
            self.notebook.tab(idx_explored, state="hidden")
            self.notebook.tab(idx_nodes, text="Neighbors")
            self.stats["Frontier"].set("N/A")
            self.stats["Explored"].set("N/A")
            self.tab_frontier.delete('1.0', tk.END)
            self.tab_explored.delete('1.0', tk.END)
            self.tab_frontier.insert('1.0', "Not applicable for local search.")
            self.tab_explored.insert('1.0', "Not applicable for local search.")
        else:
            self.notebook.tab(idx_frontier, state="normal")
            self.notebook.tab(idx_explored, state="normal")
            self.notebook.tab(idx_nodes, text="Nodes")
            self.update_explored_tab_label()

    def update_explored_tab_label(self):
        g = self.group_var.get()
        if g == 3:
            return
        self.notebook.tab(self.tab_explored, text="Explored")

    def update_algo_menu(self):
        g = self.group_var.get()
        if g == 1:
            algos = ["BFS", "DFS", "IDS"]
        elif g == 2:
            algos = ["Greedy Best-First", "A* (Misplaced)", "UCS"]
        elif g == 3:
            algos = ["Simple Hill Climbing", "Random Restart Hill Climbing",
                     "Local Beam Search", "Simulated Annealing"]
        elif g == 5:
            algos = ["Backtracking", "Forward Checking", "AC-3", "Min-Conflicts"]
        elif g == 6:
            algos = ["Minimax", "Alpha-Beta", "Expectimax"]
        else:
            algos = []
        self.algo_menu['values'] = algos
        if algos:
            self.algo_var.set(algos[0])
        else:
            self.algo_var.set("")

    # ----------- Complex frame & panel setup -----------
    def setup_complex_frame(self):
        for w in self.complex_frame.winfo_children():
            w.destroy()

        tk.Label(self.complex_frame, text="Sub-mode:", bg="#f5f6f8").pack(side=tk.LEFT, padx=(0,5))
        submode_menu = ttk.Combobox(self.complex_frame, textvariable=self.complex_submode,
                                    values=["Partial Start, Unknown Goal",
                                            "Partial Goal, Unknown Start",
                                            "Unknown Both"],
                                    state="readonly", width=25)
        submode_menu.pack(side=tk.LEFT, padx=5)
        submode_menu.bind("<<ComboboxSelected>>", self.update_complex_ui)

        tk.Label(self.complex_frame, text="DFS depth:", bg="#f5f6f8").pack(side=tk.LEFT, padx=(10,2))
        self.complex_param_entry = tk.Spinbox(self.complex_frame, from_=1, to=200, width=4)
        self.complex_param_entry.delete(0, "end"); self.complex_param_entry.insert(0, "35")
        self.complex_param_entry.pack(side=tk.LEFT, padx=2)

        btn_frame = tk.Frame(self.complex_frame, bg="#f5f6f8")
        btn_frame.pack(side=tk.LEFT, padx=10)

        self.btn_run_dfs_complex = tk.Button(btn_frame, text="Run DFS", bg="#22c55e", fg="white",
                                             bd=0, command=self.start_dfs_complex, width=10)
        self.btn_run_dfs_complex.pack(side=tk.LEFT, padx=3)

        self.btn_run_complex = tk.Button(btn_frame, text="Run AND-OR", bg="#2563eb", fg="white",
                                         bd=0, command=self.start_complex_solve, width=12)
        self.btn_run_complex.pack(side=tk.LEFT, padx=3)

        self.btn_random_complex = tk.Button(btn_frame, text="Random", bg="#ffffff", bd=1,
                                            relief=tk.SOLID, command=self.shuffle_puzzle, width=8)
        self.btn_random_complex.pack(side=tk.LEFT, padx=3)

    def setup_complex_panel(self):
        for w in self.complex_panel.winfo_children():
            w.destroy()
        ctrl = tk.Frame(self.complex_panel, bg="#f5f6f8")
        ctrl.pack(fill=tk.X, pady=2)
        tk.Label(ctrl, text="Sub-mode:", bg="#f5f6f8").pack(side=tk.LEFT, padx=5)
        submode_menu = ttk.Combobox(ctrl, textvariable=self.complex_submode,
                                    values=["Partial Start, Unknown Goal",
                                            "Partial Goal, Unknown Start",
                                            "Unknown Both"],
                                    state="readonly", width=25)
        submode_menu.pack(side=tk.LEFT, padx=5)
        submode_menu.bind("<<ComboboxSelected>>", self.update_complex_ui)
        self.complex_dynamic = tk.Frame(self.complex_panel, bg="#f5f6f8", height=150)
        self.complex_dynamic.pack(fill=tk.BOTH, expand=False, pady=5)
        self.complex_dynamic.pack_propagate(False)
        self.complex_actions = tk.Frame(self.complex_panel, bg="#f5f6f8")
        self.complex_actions.pack(fill=tk.X, pady=2)
        self.update_complex_ui()

    def update_complex_ui(self, event=None):
        for w in self.complex_dynamic.winfo_children():
            w.destroy()
        for w in self.complex_actions.winfo_children():
            w.destroy()
        submode = self.complex_submode.get()
        if "Partial Start" in submode:
            tk.Label(self.complex_dynamic, text="Start template:", bg="#f5f6f8").pack(anchor="w")
            self.template_entry = tk.Entry(self.complex_dynamic, font=("Courier", 10), width=25)
            self.template_entry.insert(0, "? 2 3 4 5 6 7 8 ?")
            self.template_entry.pack(fill=tk.X, pady=2)
            tk.Label(self.complex_dynamic, text="Goal belief:", bg="#f5f6f8").pack(anchor="w")
            self.belief_text_goal = tk.Text(self.complex_dynamic, height=3, width=30, font=("Courier", 9))
            self.belief_text_goal.pack(fill=tk.BOTH, expand=True)
            tk.Button(self.complex_actions, text="Generate Random Goals", command=self.generate_goal_belief).pack(side=tk.LEFT, padx=5)
            tk.Label(self.complex_actions, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.goal_count = tk.Spinbox(self.complex_actions, from_=1, to=10, width=3)
            self.goal_count.delete(0, "end"); self.goal_count.insert(0, "3")
            self.goal_count.pack(side=tk.LEFT)
        elif "Partial Goal" in submode:
            tk.Label(self.complex_dynamic, text="Goal template:", bg="#f5f6f8").pack(anchor="w")
            self.template_entry = tk.Entry(self.complex_dynamic, font=("Courier", 10), width=25)
            self.template_entry.insert(0, "1 2 3 4 5 6 ? 8 ?")
            self.template_entry.pack(fill=tk.X, pady=2)
            tk.Label(self.complex_dynamic, text="Start belief:", bg="#f5f6f8").pack(anchor="w")
            self.belief_text_start = tk.Text(self.complex_dynamic, height=3, width=30, font=("Courier", 9))
            self.belief_text_start.pack(fill=tk.BOTH, expand=True)
            tk.Button(self.complex_actions, text="Generate Random Starts", command=self.generate_start_belief).pack(side=tk.LEFT, padx=5)
            tk.Label(self.complex_actions, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.start_count = tk.Spinbox(self.complex_actions, from_=1, to=10, width=3)
            self.start_count.delete(0, "end"); self.start_count.insert(0, "3")
            self.start_count.pack(side=tk.LEFT)
        else:  # Unknown Both
            tk.Label(self.complex_dynamic, text="Start belief:", bg="#f5f6f8").pack(anchor="w")
            self.start_belief_text = tk.Text(self.complex_dynamic, height=2, width=30, font=("Courier", 9))
            self.start_belief_text.pack(fill=tk.BOTH, expand=True)
            tk.Label(self.complex_dynamic, text="Goal belief:", bg="#f5f6f8").pack(anchor="w")
            self.goal_belief_text = tk.Text(self.complex_dynamic, height=2, width=30, font=("Courier", 9))
            self.goal_belief_text.pack(fill=tk.BOTH, expand=True)
            act = tk.Frame(self.complex_actions, bg="#f5f6f8")
            act.pack(fill=tk.X)
            tk.Button(act, text="Generate Random Starts", command=self.generate_start_belief).pack(side=tk.LEFT, padx=5)
            tk.Label(act, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.start_count = tk.Spinbox(act, from_=1, to=10, width=3)
            self.start_count.delete(0, "end"); self.start_count.insert(0, "2")
            self.start_count.pack(side=tk.LEFT)
            tk.Button(act, text="Generate Random Goals", command=self.generate_goal_belief).pack(side=tk.LEFT, padx=10)
            tk.Label(act, text="Count:", bg="#f5f6f8").pack(side=tk.LEFT)
            self.goal_count = tk.Spinbox(act, from_=1, to=10, width=3)
            self.goal_count.delete(0, "end"); self.goal_count.insert(0, "2")
            self.goal_count.pack(side=tk.LEFT)

    def generate_start_belief(self):
        try: n = int(self.start_count.get())
        except: n = 3
        states = generate_random_solvable_states(n)
        self.complex_initial_belief = states
        if hasattr(self, 'belief_text_start'):
            self.belief_text_start.delete('1.0', tk.END)
            for s in states: self.belief_text_start.insert(tk.END, f"{s}\n")
        elif hasattr(self, 'start_belief_text'):
            self.start_belief_text.delete('1.0', tk.END)
            for s in states: self.start_belief_text.insert(tk.END, f"{s}\n")

    def generate_goal_belief(self):
        try: n = int(self.goal_count.get())
        except: n = 3
        states = generate_random_solvable_states(n)
        self.complex_goal_states = states
        if hasattr(self, 'belief_text_goal'):
            self.belief_text_goal.delete('1.0', tk.END)
            for s in states: self.belief_text_goal.insert(tk.END, f"{s}\n")
        elif hasattr(self, 'goal_belief_text'):
            self.goal_belief_text.delete('1.0', tk.END)
            for s in states: self.goal_belief_text.insert(tk.END, f"{s}\n")

    # --- Run DFS in complex mode ---
    def start_dfs_complex(self):
        start_state = tuple(self.current_state)
        Node._id_counter = 65
        try:
            param_val = int(self.complex_param_entry.get())
        except:
            param_val = 35
        self.clear_tabs()
        self.btn_run_dfs_complex.config(state=tk.DISABLED)
        self.btn_run_complex.config(state=tk.DISABLED)
        threading.Thread(target=self.solve, args=("DFS", start_state, param_val), daemon=True).start()

    def start_complex_solve(self):
        submode = self.complex_submode.get()
        if "Partial Start" in submode:
            start_tmpl = self.template_entry.get()
            start_states = parse_template(start_tmpl)
            if not start_states:
                messagebox.showwarning("Error", "Invalid start template.")
                return
            initial_belief = start_states
            goal_states = self.complex_goal_states if self.complex_goal_states else [GOAL]
        elif "Partial Goal" in submode:
            goal_tmpl = self.template_entry.get()
            goal_states = parse_template(goal_tmpl)
            if not goal_states:
                messagebox.showwarning("Error", "Invalid goal template.")
                return
            initial_belief = [tuple(self.current_state)]
            goal_states = goal_states
        else:
            initial_belief = self.complex_initial_belief
            goal_states = self.complex_goal_states
            if not initial_belief or not goal_states:
                messagebox.showwarning("Error", "Please generate belief start and goal.")
                return

        def actions(state): return [a for _, a in get_neighbors(state)]
        def results(state, action):
            for nstate, act in get_neighbors(state):
                if act == action: return [nstate]
            return []

        problem = ComplexProblem(initial_belief, lambda s: s in goal_states, actions, results)
        self.clear_tabs()
        self.update_stat("Status", "Running AND-OR Graph Search (depth limit 50)...")
        self.btn_run_complex.config(state=tk.DISABLED)
        self.btn_run_dfs_complex.config(state=tk.DISABLED)
        threading.Thread(target=self._complex_solve_thread, args=(problem,), daemon=True).start()

    def _complex_solve_thread(self, problem):
        start_time = time.time()
        def log(msg):
            self.write_to_tab(self.tab_steplog, msg)
        plan = and_or_graph_search(problem, max_depth=50, log_func=log)
        elapsed = time.time() - start_time
        if plan:
            self.conditional_plan = plan
            self.tab_conditional.delete('1.0', tk.END)
            self.tab_conditional.insert('1.0', format_plan(plan))
            self.update_stat("Status", f"Found plan in {elapsed:.4f}s")
        else:
            self.tab_conditional.insert('1.0', "No plan found (failure).")
            self.update_stat("Status", "No plan found.")
        self.root.after(0, lambda: self.btn_run_complex.config(state=tk.NORMAL))
        self.root.after(0, lambda: self.btn_run_dfs_complex.config(state=tk.NORMAL))

    # ----------- Common helpers -----------
    def update_grid(self):
        for i, val in enumerate(self.current_state):
            if val == 0:
                self.cells[i].config(text="", bg="#e2e8f0")
            else:
                self.cells[i].config(text=str(val), bg="#2563eb")

    def reset_puzzle(self):
        self.current_state = list(GOAL)
        self.path = []
        self.current_step_idx = 0
        self.is_running = False
        self.btn_auto.config(text="Auto Play", bg="#ffffff")
        self.update_grid()
        self.clear_tabs()
        for key, var in self.stats.items():
            var.set("-" if key != "Status" else "Ready.")
        Node._id_counter = 65

    def clear_tabs(self):
        for tab in [self.tab_nodes, self.tab_frontier, self.tab_explored, self.tab_steplog,
                    self.tab_conditional, self.tab_solution]:
            tab.delete('1.0', tk.END)

    def shuffle_puzzle(self):
        self.reset_puzzle()
        state = list(GOAL)
        for _ in range(14):
            neighbors = get_neighbors(tuple(state))
            state = list(random.choice(neighbors)[0])
        self.current_state = state
        self.update_grid()

    def write_to_tab(self, tab, text):
        self.root.after(0, lambda: self._safe_write(tab, text))

    def _safe_write(self, tab, text):
        tab.insert(tk.END, text + "\n")
        tab.see(tk.END)

    def update_stat(self, key, value):
        self.root.after(0, lambda: self.stats[key].set(str(value)))

    def write_frontier_explored(self, frontier_list, explored_set, mode="explored"):
        if self.group_var.get() == 3:
            return
        def format_state_list(state_list):
            if not state_list:
                return "(empty)"
            return "\n".join(format_state_matrix(s) for s in state_list)

        self.tab_frontier.delete('1.0', tk.END)
        if frontier_list:
            states_f = [n.state for n in frontier_list]
            self.tab_frontier.insert('1.0', f"Frontier ({len(states_f)} states):\n")
            for s in states_f[:20]:
                self.tab_frontier.insert('1.0', format_state_matrix(s) + "\n")
            if len(states_f) > 20:
                self.tab_frontier.insert('1.0', f"... and {len(states_f)-20} more\n")
        else:
            self.tab_frontier.insert('1.0', "Frontier is empty.")

        label = "Reached" if mode == "reached" else "Explored"
        self.tab_explored.delete('1.0', tk.END)
        if explored_set:
            if isinstance(explored_set, set):
                states_e = list(explored_set)
            else:
                states_e = list(explored_set)
            self.tab_explored.insert('1.0', f"{label} ({len(states_e)} states):\n")
            for s in states_e[:20]:
                self.tab_explored.insert('1.0', format_state_matrix(s) + "\n")
            if len(states_e) > 20:
                self.tab_explored.insert('1.0', f"... and {len(states_e)-20} more\n")
        else:
            self.tab_explored.insert('1.0', f"{label} is empty.")

    # ----------- Solution Path display & navigation -----------
    def display_solution_path(self):
        self.tab_solution.delete('1.0', tk.END)
        if not self.path:
            return
        for idx, node in enumerate(self.path):
            action = node.action if node.action else "Start"
            self.tab_solution.insert(tk.END, f"Step {idx}: {action}\n")
            self.tab_solution.insert(tk.END, format_state_matrix(node.state) + "\n\n")
        self.highlight_current_step()

    def highlight_current_step(self):
        self.tab_solution.tag_remove("highlight", "1.0", tk.END)
        if self.path and 0 <= self.current_step_idx < len(self.path):
            line_start = f"Step {self.current_step_idx}:"
            start_pos = self.tab_solution.search(line_start, "1.0", stopindex=tk.END)
            if start_pos:
                end_pos = f"{start_pos}+{len(line_start)}c"
                self.tab_solution.tag_add("highlight", start_pos, end_pos)
                self.tab_solution.see(start_pos)

    def on_solution_click(self, event):
        index = self.tab_solution.index(f"@{event.x},{event.y}")
        line_start = self.tab_solution.index(f"{index} linestart")
        line_text = self.tab_solution.get(line_start, f"{line_start} lineend")
        if line_text.startswith("Step "):
            try:
                step_num = int(line_text.split()[1].rstrip(':'))
                if 0 <= step_num < len(self.path):
                    self.jump_to_step(step_num)
            except ValueError:
                pass

    def jump_to_step(self, step_idx):
        if not self.path or step_idx < 0 or step_idx >= len(self.path):
            return
        self.current_step_idx = step_idx
        node = self.path[step_idx]
        self.current_state = list(node.state)
        self.update_grid()
        self.update_stat("Step", f"{step_idx} / {len(self.path)-1}")
        self.update_stat("Action", node.action if node.action else "Start")
        self.update_stat("Path Cost", node.cost)
        self.update_stat("Depth", node.depth)
        self.highlight_current_step()

    # ----------- Start standard algorithms -----------
    def start_solve(self):
        if self.group_var.get() == 4:
            return
        algo = self.algo_var.get()
        start_state = tuple(self.current_state)
        Node._id_counter = 65
        try:
            param_val = int(self.param_entry.get())
        except ValueError:
            param_val = 35
        self.clear_tabs()
        self.btn_run.config(state=tk.DISABLED)
        threading.Thread(target=self.solve, args=(algo, start_state, param_val), daemon=True).start()

    def solve(self, algo, start, param_val):
        start_time = time.time()
        result_node = None
        root_node = Node(start, cost=0, depth=0, h_cost=heuristic(start))
        explored = set()
        frontier_list = []

        # ======================== GROUP 1: Uninformed ========================
        if algo == "BFS":
            self.update_stat("Status", "BFS (FIFO Queue)")
            queue = collections.deque([root_node])
            explored.add(start)
            while queue:
                node = queue.popleft()
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                for state, action in get_neighbors(node.state):
                    if state not in explored and node.depth < param_val:
                        explored.add(state)
                        child = Node(state, node, action, node.cost + 1, node.depth + 1)
                        queue.append(child)
                        self.write_to_tab(self.tab_nodes,
                                          f"{child.name} | parent={node.name} | action={action} | depth={child.depth}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(queue)} node(s)")
                self.update_stat("Explored", f"{len(explored)} state(s)")
            frontier_list = list(queue)

        elif algo == "DFS":
            self.update_stat("Status", "DFS (LIFO Stack)")
            stack = [root_node]
            explored.add(start)
            while stack:
                node = stack.pop()
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                if node.depth < param_val:
                    for state, action in get_neighbors(node.state):
                        if state not in explored:
                            explored.add(state)
                            child = Node(state, node, action, node.cost + 1, node.depth + 1)
                            stack.append(child)
                            self.write_to_tab(self.tab_nodes,
                                              f"{child.name} | parent={node.name} | action={action} | depth={child.depth}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(stack)} node(s)")
                self.update_stat("Explored", f"{len(explored)} state(s)")
            frontier_list = stack

        elif algo == "IDS":
            self.update_stat("Status", "Iterative Deepening Search")
            max_depth_limit = param_val
            final_explored = set()
            for depth_limit in range(1, max_depth_limit + 1):
                self.update_stat("IDS Limit", depth_limit)
                self.write_to_tab(self.tab_steplog, f"Trying depth limit = {depth_limit}")
                visited_ids = {start}
                stack = [(root_node, 0)]
                found = False
                while stack:
                    node, d = stack.pop()
                    if node.state == GOAL:
                        result_node = node; found = True; break
                    if d < depth_limit:
                        for state, action in get_neighbors(node.state):
                            if state not in visited_ids:
                                visited_ids.add(state)
                                child = Node(state, node, action, node.cost + 1, d + 1)
                                stack.append((child, d + 1))
                                self.write_to_tab(self.tab_nodes,
                                                  f"{child.name} | parent={node.name} | action={action} | d={d+1}")
                final_explored = visited_ids
                if found:
                    break
            explored = final_explored
            frontier_list = [n for n, _ in stack] if not result_node else []

        # ======================== GROUP 2: Informed ========================
        elif algo == "Greedy Best-First":
            self.update_stat("Status", "Greedy Best-First (h only)")
            frontier = [(root_node.h_cost, id(root_node), root_node)]
            reached = {start: root_node}
            while frontier:
                _, _, node = heapq.heappop(frontier)
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                for state, action in get_neighbors(node.state):
                    child = Node(state, node, action, node.cost + 1, node.depth + 1,
                                 h_cost=heuristic(state))
                    if state not in reached:
                        reached[state] = child
                        heapq.heappush(frontier, (child.h_cost, id(child), child))
                        self.write_to_tab(self.tab_nodes,
                                          f"{child.name} | parent={node.name} | h={child.h_cost}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(frontier)} node(s)")
                self.update_stat("Explored", f"{len(reached)} state(s)")
            frontier_list = [n for _, _, n in frontier]
            explored = set(reached.keys())

        elif algo == "A* (Misplaced)":
            self.update_stat("Status", "A* (f = g + h)")
            frontier = [(root_node.f_cost, id(root_node), root_node)]
            reached = {start: root_node}
            while frontier:
                _, _, node = heapq.heappop(frontier)
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                for state, action in get_neighbors(node.state):
                    g_new = node.cost + 1
                    h_val = heuristic(state)
                    if state in reached:
                        old_node = reached[state]
                        if g_new < old_node.cost:
                            old_node.cost = g_new
                            old_node.h_cost = h_val
                            old_node.f_cost = g_new + h_val
                            old_node.parent = node
                            old_node.action = action
                            old_node.depth = node.depth + 1
                            heapq.heappush(frontier, (old_node.f_cost, id(old_node), old_node))
                    else:
                        child = Node(state, node, action, g_new, node.depth + 1, h_val)
                        reached[state] = child
                        heapq.heappush(frontier, (child.f_cost, id(child), child))
                        self.write_to_tab(self.tab_nodes,
                                          f"{child.name} | g={child.cost} h={child.h_cost} f={child.f_cost}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(frontier)} node(s)")
                self.update_stat("Explored", f"{len(reached)} state(s)")
            frontier_list = [n for _, _, n in frontier]
            explored = set(reached.keys())

        elif algo == "UCS":
            self.update_stat("Status", "UCS (Uniform Cost)")
            pq = [(root_node.cost, id(root_node), root_node)]
            explored.add(start)
            while pq:
                cost, _, node = heapq.heappop(pq)
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                if node.state not in explored:
                    explored.add(node.state)
                    if node.depth < param_val:
                        for state, action in get_neighbors(node.state):
                            if state not in explored:
                                child = Node(state, node, action, node.cost + 1, node.depth + 1)
                                heapq.heappush(pq, (child.cost, id(child), child))
                                self.write_to_tab(self.tab_nodes,
                                                  f"{child.name} | cost={child.cost}")
                self.update_stat("Frontier", f"{len(pq)} node(s)")
                self.update_stat("Explored", f"{len(explored)} state(s)")
            frontier_list = [n for _, _, n in pq]

        # ======================== GROUP 3: Local Search ========================
        elif algo == "Simple Hill Climbing":
            self.update_stat("Status", "Simple Hill Climbing")
            current_node = root_node
            self.write_to_tab(self.tab_steplog,
                              f"Start: {format_state_matrix(current_node.state)} (Value={value(current_node.state)})")
            while True:
                if current_node.state == GOAL:
                    result_node = current_node; break
                found_better = False
                for state, action in get_neighbors(current_node.state):
                    self.write_to_tab(self.tab_nodes,
                                      f"Neighbor: action={action}, value={value(state)}\n{format_state_matrix(state)}")
                    if value(state) > value(current_node.state):
                        current_node = Node(state, current_node, action, current_node.cost+1, current_node.depth+1)
                        self.write_to_tab(self.tab_steplog,
                                          f"  -> Move to (Value={value(state)})")
                        found_better = True
                        break
                if not found_better:
                    self.write_to_tab(self.tab_steplog, f"Local maximum at {current_node.name}")
                    result_node = current_node; break

        elif algo == "Random Restart Hill Climbing":
            max_restart = param_val
            self.update_stat("Status", "Random Restart Hill Climbing")
            self.write_to_tab(self.tab_steplog, f"Max restarts = {max_restart}")
            success = False
            for i in range(1, max_restart+1):
                self.write_to_tab(self.tab_steplog, f"Restart {i}")
                if i == 1:
                    current_state_tuple = start
                else:
                    temp = list(GOAL)
                    for _ in range(18): temp = list(random.choice(get_neighbors(tuple(temp)))[0])
                    current_state_tuple = tuple(temp)
                current_node = Node(current_state_tuple, parent=None if i==1 else root_node, action=f"Restart {i}")
                self.write_to_tab(self.tab_steplog,
                                  f"Current state (Value={value(current_node.state)}):\n{format_state_matrix(current_node.state)}")
                while True:
                    if current_node.state == GOAL:
                        result_node = current_node; success = True; break
                    better = []
                    for state, action in get_neighbors(current_node.state):
                        self.write_to_tab(self.tab_nodes,
                                          f"Neighbor: {action} value={value(state)}\n{format_state_matrix(state)}")
                        if value(state) > value(current_node.state):
                            better.append((state, action))
                    if not better:
                        self.write_to_tab(self.tab_steplog, "Stuck – restarting.")
                        break
                    better.sort(key=lambda x: value(x[0]), reverse=True)
                    best_state, best_action = better[0]
                    current_node = Node(best_state, current_node, best_action, current_node.cost+1, current_node.depth+1)
                    self.write_to_tab(self.tab_steplog,
                                      f"  -> Selected best (Value={value(best_state)})")
                if success: break
            if not success:
                self.write_to_tab(self.tab_steplog, "Failed all restarts!")

        elif algo == "Local Beam Search":
            k = param_val
            self.update_stat("Status", "Local Beam Search")
            self.write_to_tab(self.tab_steplog, f"Beam width k = {k}")
            current_states = [root_node]
            for _ in range(k - 1):
                rand_state = list(start)
                for _ in range(10):
                    nbs = get_neighbors(tuple(rand_state))
                    rand_state = list(random.choice(nbs)[0])
                current_states.append(Node(tuple(rand_state)))
            self.write_to_tab(self.tab_steplog, f"Initial beam ({k} states):")
            for ns in current_states:
                self.write_to_tab(self.tab_steplog, f"  {ns.name} (Value={value(ns.state)})")
            while True:
                all_neighbors = []
                found_goal = None
                for node in current_states:
                    for state, action in get_neighbors(node.state):
                        child = Node(state, node, action, node.cost+1, node.depth+1)
                        all_neighbors.append(child)
                        if state == GOAL:
                            found_goal = child; break
                    if found_goal: break
                if found_goal:
                    result_node = found_goal; break
                if not all_neighbors:
                    break
                all_neighbors.sort(key=lambda x: value(x.state), reverse=True)
                current_states = all_neighbors[:k]
                self.write_to_tab(self.tab_steplog, f"Beam after iteration (top {k}):")
                for n in current_states:
                    self.write_to_tab(self.tab_steplog, f"  {n.name} (Value={value(n.state)})")
                self.write_to_tab(self.tab_nodes, "All generated neighbors:")
                for n in all_neighbors:
                    self.write_to_tab(self.tab_nodes,
                                      f"  {n.name} (Value={value(n.state)}) action={n.action}\n{format_state_matrix(n.state)}")

        elif algo == "Simulated Annealing":
            self.update_stat("Status", "Simulated Annealing")
            T0 = 10.0
            Tmin = 0.1
            alpha = 0.99
            max_iter = param_val
            current_state = start
            current_h = heuristic(current_state)
            current_node = root_node
            self.write_to_tab(self.tab_steplog,
                              f"Start (h={current_h}) T0={T0}, Tmin={Tmin}, alpha={alpha}, max iter={max_iter}")
            self.write_to_tab(self.tab_steplog, f"Initial state:\n{format_state_matrix(current_state)}")
            T = T0
            for it in range(max_iter):
                if current_state == GOAL:
                    result_node = current_node; break
                neighbors = get_neighbors(current_state)
                next_state, action = random.choice(neighbors)
                next_h = heuristic(next_state)
                delta = next_h - current_h
                accept = False
                if delta <= 0:
                    accept = True
                    reason = "better"
                else:
                    p = math.exp(-delta / T)
                    if random.random() < p:
                        accept = True
                        reason = f"accepted with p={p:.4f}"
                    else:
                        reason = f"rejected, p={p:.4f}"
                self.write_to_tab(self.tab_nodes,
                                  f"Iter {it}: T={T:.4f}, neighbor {action}, h={next_h} (Δ={delta:+d}) -> {reason}\n{format_state_matrix(next_state)}")
                if accept:
                    current_state = next_state
                    current_h = next_h
                    current_node = Node(current_state, current_node, action, current_node.cost+1, current_node.depth+1)
                T *= alpha
                if T < Tmin:
                    self.write_to_tab(self.tab_steplog, f"Temperature below Tmin, stopping.")
                    break
            if not result_node and current_state == GOAL:
                result_node = current_node

        # ======================== GROUP 5: CSP ========================
        elif algo in ("Backtracking", "Forward Checking", "AC-3", "Min-Conflicts"):
            self.update_stat("Status", algo)
            bfs_q = collections.deque([root_node])
            bfs_vis = {start}
            while bfs_q:
                node = bfs_q.popleft()
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                for state, action in get_neighbors(node.state):
                    if state not in bfs_vis:
                        bfs_vis.add(state)
                        child = Node(state, node, action, node.cost+1, node.depth+1)
                        bfs_q.append(child)
                        # --- ADDED logging to Nodes tab ---
                        self.write_to_tab(self.tab_nodes,
                                          f"{child.name} | parent={node.name} | action={action} | depth={child.depth}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(bfs_q)} node(s)")
                self.update_stat("Explored", f"{len(bfs_vis)} state(s)")
            explored = bfs_vis
            frontier_list = list(bfs_q)

        # ======================== GROUP 6: Adversarial ========================
        elif algo in ("Minimax", "Alpha-Beta", "Expectimax"):
            self.update_stat("Status", algo)
            frontier_a = [(root_node.f_cost, id(root_node), root_node)]
            reached_a = {start: root_node}
            while frontier_a:
                _, _, node = heapq.heappop(frontier_a)
                self.update_stat("Expanded", node.name)
                if node.state == GOAL:
                    result_node = node; break
                if node.state not in reached_a:
                    reached_a[node.state] = node
                for state, action in get_neighbors(node.state):
                    if state not in reached_a:
                        h = heuristic(state)
                        child = Node(state, node, action, node.cost+1, node.depth+1, h)
                        reached_a[state] = child
                        heapq.heappush(frontier_a, (child.f_cost, id(child), child))
                        # --- ADDED logging to Nodes tab ---
                        self.write_to_tab(self.tab_nodes,
                                          f"{child.name} | g={child.cost} h={child.h_cost} f={child.f_cost} | action={action}\n{format_state_matrix(state)}")
                self.update_stat("Frontier", f"{len(frontier_a)} node(s)")
                self.update_stat("Explored", f"{len(reached_a)} state(s)")
            explored = set(reached_a.keys())
            frontier_list = [n for _, _, n in frontier_a]

        # ----------- If local search found a goal, use BFS to get clean path -----------
        if result_node and algo in ("Simple Hill Climbing", "Random Restart Hill Climbing",
                                    "Local Beam Search", "Simulated Annealing"):
            bfs_q = collections.deque([root_node])
            bfs_vis = {start}
            while bfs_q:
                node = bfs_q.popleft()
                if node.state == GOAL:
                    result_node = node; break
                for state, action in get_neighbors(node.state):
                    if state not in bfs_vis:
                        bfs_vis.add(state)
                        bfs_q.append(Node(state, node, action, node.cost+1, node.depth+1))

        self.write_frontier_explored(frontier_list, explored)

        # ----------- Final result processing -----------
        elapsed = time.time() - start_time
        if result_node:
            self.path = []
            curr = result_node
            while curr:
                self.path.append(curr)
                curr = curr.parent
            self.path.reverse()
            self.current_step_idx = 0
            self.current_state = list(self.path[0].state)
            self.update_grid()
            self.update_stat("Algorithm", algo)
            self.update_stat("Path Cost", result_node.cost)
            self.update_stat("Depth", result_node.depth)
            self.update_stat("Step", f"0 / {len(self.path)-1}")
            self.update_stat("Action", "Start Position")
            self.root.after(0, self.display_solution_path)
            self.root.after(500, self.start_auto_play)
            self.update_stat("Status", f"Found in {elapsed:.4f}s")
        else:
            self.update_stat("Status", "No path found!")

        # Re-enable buttons
        self.root.after(0, lambda: self.btn_run.config(state=tk.NORMAL))
        if hasattr(self, 'btn_run_dfs_complex'):
            self.root.after(0, lambda: self.btn_run_dfs_complex.config(state=tk.NORMAL))
        if hasattr(self, 'btn_run_complex'):
            self.root.after(0, lambda: self.btn_run_complex.config(state=tk.NORMAL))

    def start_auto_play(self):
        if self.path and not self.is_running:
            self.toggle_auto()

    # ----------- Step navigation & auto play -----------
    def prev_step(self):
        if not self.path or self.current_step_idx <= 0: return
        self.current_step_idx -= 1
        node = self.path[self.current_step_idx]
        self.current_state = list(node.state)
        self.update_grid()
        self.update_stat("Step", f"{self.current_step_idx} / {len(self.path)-1}")
        self.update_stat("Action", node.action if node.action else "Start")
        self.update_stat("Path Cost", node.cost)
        self.update_stat("Depth", node.depth)
        self.highlight_current_step()

    def next_step(self):
        if not self.path or self.current_step_idx >= len(self.path) - 1: return
        self.current_step_idx += 1
        node = self.path[self.current_step_idx]
        self.current_state = list(node.state)
        self.update_grid()
        self.update_stat("Step", f"{self.current_step_idx} / {len(self.path)-1}")
        self.update_stat("Action", node.action)
        self.update_stat("Path Cost", node.cost)
        self.update_stat("Depth", node.depth)
        self.highlight_current_step()

    def toggle_auto(self):
        if self.is_running:
            self.is_running = False
            self.btn_auto.config(text="Auto Play", bg="#ffffff")
        else:
            if not self.path: return
            self.is_running = True
            self.btn_auto.config(text="Stop", bg="#ef4444", fg="white")
            threading.Thread(target=self.auto_loop, daemon=True).start()

    def auto_loop(self):
        while self.is_running and self.current_step_idx < len(self.path) - 1:
            self.root.after(0, self.next_step)
            time.sleep(self.speed_scale.get() / 1000.0)
        self.is_running = False
        self.root.after(0, lambda: self.btn_auto.config(text="Auto Play", bg="#ffffff", fg="black"))

if __name__ == "__main__":
    root = tk.Tk()
    app = EightPuzzleGUI(root)
    root.mainloop()